In [1]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3


Found existing installation: protobuf 6.33.0
Uninstalling protobuf-6.33.0:
  Successfully uninstalled protobuf-6.33.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; pyth

In [2]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, utils as vutils
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import Subset
from transformers import BertTokenizer, BertModel
from PIL import Image
from tqdm import tqdm
import os
from torchvision.models import resnet50
from torchvision import models
from collections import Counter
import re
import glob

SEED = 69
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

IMAGE_DIR = '../input/vqa-fr/images'
QUESTIONS_DIR = '../input/vqa-fr/questions'
ANNOTATIONS_DIR = '../input/vqa-fr/annotations'
REDUCED_TRAINING = True
REDUCED_COUNT = 2000
# amount of imgs used

def default_tokenizer(sentence):

    regex = re.compile(r'(\W+)')
    tokens = regex.split(sentence.lower())
    tokens = [w.strip() for w in tokens if len(w.strip()) > 0]
    return tokens

2025-12-14 02:25:10.322919: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765679110.505165      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765679110.555616      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
import json
from collections import Counter
import os


annotation_dir = "../input/vqa-fr/annotations"
output_dir = "data"
top_k = 2500  # top N answers to keep


train_ann_file = os.path.join(annotation_dir, "v2_mscoco_train2014_annotations.json")
with open(train_ann_file, 'r') as f:
    train_data = json.load(f)

annotations = train_data['annotations']

# Count all multiple-choice answers
all_answers = [ann['multiple_choice_answer'] for ann in annotations]
answer_counter = Counter(all_answers)

# Keep top-k most frequent answers
top_answers = [ans for ans, _ in answer_counter.most_common(top_k)]

# Save to annotation_vocabs.txt
os.makedirs(output_dir, exist_ok=True)

vocab_file = os.path.join(output_dir, "annotation_vocabs.txt")
with open(vocab_file, 'w') as f:
    f.write('<pad>\n')
    f.write('<unk>\n')
    for ans in top_answers:
        f.write(ans + '\n')

print(f"Saved top {top_k} answers to {vocab_file}")


question_files = [
    os.path.join(QUESTIONS_DIR, "v2_OpenEnded_mscoco_train2014_questions.json"),
    os.path.join(QUESTIONS_DIR, "v2_OpenEnded_mscoco_val2014_questions.json")
]

all_tokens = []

for question_file in question_files:
    with open(question_file, 'r') as f:
        data = json.load(f)
        questions = data['questions']

        for qu in questions:
            tokens = default_tokenizer(qu['question'])
            all_tokens.extend(tokens)

# Count all tokens
token_counter = Counter(all_tokens)

# Keep top-k most frequent tokens
top_tokens = [token for token, _ in token_counter.most_common(top_k)]

# Save to question_vocabs.txt
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, 'processed_w_bert'), exist_ok=True)
os.makedirs(os.path.join(output_dir, 'regular'), exist_ok=True)

vocab_file = os.path.join(output_dir, "question_vocabs.txt")

with open(vocab_file, 'w') as f:
    # Add special tokens first
    f.write('<pad>\n')
    f.write('<unk>\n')
    for token in top_tokens:
        f.write(token + '\n')

print(f"Saved top {top_k} question tokens (+ 2 special tokens) to {vocab_file}")
print(f"Total unique tokens found: {len(token_counter)}")


Saved top 2500 answers to data/annotation_vocabs.txt
Saved top 2500 question tokens (+ 2 special tokens) to data/question_vocabs.txt
Total unique tokens found: 15334


In [4]:
output_dir = "data"

def match_top_ans(annotation_ans):
    annotation_dir = os.path.join(output_dir, 'annotation_vocabs.txt')
    if "top_ans" not in match_top_ans.__dict__:
        with open(annotation_dir, 'r') as f:
            match_top_ans.top_ans = {line.strip() for line in f}
    annotation_ans = {ans['answer'] for ans in annotation_ans }
    valid_ans = annotation_ans & match_top_ans.top_ans

    if len(valid_ans) == 0:
        valid_ans = ['<unk>']
        match_top_ans.unk_ans += 1
    return valid_ans, annotation_ans

# adds transformation to imgs, resize em
def preprocessing(question_dir, annotations_dir, image_dir, labeled, tokenizer=None, max_length=32):
    with open(question_dir, 'r') as f:
        data = json.load(f)
        questions = data['questions']
        if data['data_subtype'] == 'test-dev2015':
            file_name = 'test2015'
        else:
            file_name = data['data_subtype']
    if labeled:
        template = os.path.join(annotations_dir, f"*{file_name}*.json")  # Fixed
        annotation_path = glob.glob(template)[0]
        with open(annotation_path, 'r') as f:
            annotations = json.load(f)['annotations']
        questions_dict = {ans['question_id']: ans for ans in annotations}

    match_top_ans.unk_ans = 0
    dataset = [None]*len(questions)
    for idx, qu in enumerate(questions):
        if (idx + 1)  % 50000 == 0:
            print(f'processing {data["data_subtype"]} data: {idx+1}/{len(questions)}')
        question_id = qu['question_id']
        question_itself = qu['question']
        if tokenizer:
            question_tokens = tokenizer(
                question_itself,
                # return_tensors='pt',
                padding='max_length',
                truncation=True,
                max_length= max_length)

        else:
            question_tokens = default_tokenizer(question_itself)
        img_id = qu['image_id']
        img_name = 'COCO_' + file_name + '_{:0>12d}.jpg'.format(img_id)
        img_path = os.path.join(image_dir, file_name, img_name)

        info = {'img_name': img_name,
            'img_path': img_path,
            'qu_sentence': question_itself,
            'qu_tokens': question_tokens,
            'qu_id': question_id}
        if labeled:
            annotation_ans = questions_dict[question_id]['answers']
            info['valid'], info['all_ans'] = match_top_ans(annotation_ans)
        dataset[idx] = info
    return dataset
#

processed_data = {}
processed_data_with_bert = {}
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

for file in os.listdir(QUESTIONS_DIR):
    datatype = file[20:-19]
    labeled = False if "test" in datatype else True
    question = os.path.join(QUESTIONS_DIR, file)
    processed_data[datatype] = preprocessing(question, ANNOTATIONS_DIR, IMAGE_DIR, labeled)
    processed_data_with_bert[datatype] = preprocessing(question, ANNOTATIONS_DIR, IMAGE_DIR, labeled, tokenizer=bert_tokenizer)

for key, value in processed_data.items():
    np.save(os.path.join(os.path.join(output_dir, 'regular'), f'{key}.npy'), np.array(value))
for key, value in processed_data_with_bert.items():
    np.save(os.path.join(os.path.join(output_dir, 'processed_w_bert'), f'{key}.npy'), np.array(value))




tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

processing train2014 data: 50000/443757
processing train2014 data: 100000/443757
processing train2014 data: 150000/443757
processing train2014 data: 200000/443757
processing train2014 data: 250000/443757
processing train2014 data: 300000/443757
processing train2014 data: 350000/443757
processing train2014 data: 400000/443757
processing train2014 data: 50000/443757
processing train2014 data: 100000/443757
processing train2014 data: 150000/443757
processing train2014 data: 200000/443757
processing train2014 data: 250000/443757
processing train2014 data: 300000/443757
processing train2014 data: 350000/443757
processing train2014 data: 400000/443757
processing test2015 data: 50000/447793
processing test2015 data: 100000/447793
processing test2015 data: 150000/447793
processing test2015 data: 200000/447793
processing test2015 data: 250000/447793
processing test2015 data: 300000/447793
processing test2015 data: 350000/447793
processing test2015 data: 400000/447793
processing test2015 data: 5

In [5]:
class Vocab:
    def __init__(self, vocab_file):
        vocab = self.load_vocab(vocab_file)
        self.vocab2idx = {vocab: idx for idx, vocab in enumerate(vocab)}
        self.vocab_size = len(vocab)

    def load_vocab(self, vocab_file):
        with open(vocab_file) as f:
            vocab = [v.strip() for v in f]
        return vocab

    def word2idx(self, word):
        if word in self.vocab2idx:
            return self.vocab2idx[word]
        else:
            return self.vocab2idx['<unk>']

    def idx2word(self, idx):
        return self.vocab[idx]


In [6]:
from PIL import Image
class VQADataset(Dataset):
    def __init__(self, input_dir, input_file, answer_to_id=None,
                 transform=None, max_length=32):

        self.input_data = np.load(os.path.join(input_dir, input_file), allow_pickle=True)
        self.max_length = max_length
        self.answer_to_id = answer_to_id
        self.transform = transform
        self.qu_vocab = Vocab(os.path.join(input_dir, 'question_vocabs.txt'))
        self.ans_vocab = Vocab(os.path.join(input_dir, 'annotation_vocabs.txt'))
        self.labeled = True if 'test' not in input_file else False

    def __getitem__(self, idx):
        path = self.input_data[idx]['img_path']
        img = Image.open(path).convert('RGB')
        qu_id = self.input_data[idx]['qu_id']
        qu_tokens = self.input_data[idx]['qu_tokens']
        qu2idx = np.array([self.qu_vocab.word2idx('<pad>')] * self.max_length)
        qu2idx[:len(qu_tokens)] = [self.qu_vocab.word2idx(token) for token in qu_tokens]
        sample = {'image': img, 'question': qu2idx, 'question_id': qu_id}

        if self.labeled:
            ans2idx = [self.ans_vocab.word2idx(ans) for ans in self.input_data[idx]['valid']]
            ans2idx = np.random.choice(ans2idx)
            sample['answer'] = ans2idx

        if self.transform:
            sample['image'] = self.transform(sample['image'])

        return sample

    def __len__(self):
        return len(self.input_data)



In [7]:
def data_loader(input_dir, npy_dir, batch_size, num_worker, max_length=32):

    transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
    # apparently this is better cuz resnet weights are pretrained on imageNet
    ])

    vqa_dataset = {
        'train': VQADataset(input_dir=input_dir, input_file=f'{npy_dir}/train.npy', transform=transform, max_length=max_length),
        'val':  VQADataset(input_dir=input_dir, input_file=f'{npy_dir}/val.npy', transform=transform, max_length=max_length),
    }
    data_loader = {
        key: DataLoader(
            dataset=vqa_dataset[key],
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_worker)
        for key in ['train', 'val']
    }
    return data_loader

#### Model building + Feature Extractor

---





##### Baseline (pipeline from paper)

In [8]:
class BaseLineImageEncoder(nn.Module):

  def __init__(self, embed_dim):
    super(BaseLineImageEncoder, self).__init__()

    vgg = models.vgg19(pretrained=True)
    in_features = vgg.classifier[-1].in_features  # 4096
    vgg.classifier = nn.Sequential(*list(vgg.classifier.children())[:-1])

    self.model = vgg  # Keep the entire VGG (features + classifier without last layer)
    self.fc = nn.Linear(in_features, embed_dim)

  def forward(self, x):
    with torch.no_grad():
      img_feature = self.model(x)  # Now this works - full VGG forward pass

    img_feature = self.fc(img_feature)
    l2_norm = nn.functional.normalize(img_feature, p=2, dim=1)
    return l2_norm


class BaseTextEncoder(nn.Module):
  def __init__(self, que_vocab_size, embedding_dim, hidden_size, num_hidden, qu_feature_size):

    super(BaseTextEncoder, self).__init__()
    self.embed_word = nn.Embedding(que_vocab_size, embedding_dim)
    self.tanh = nn.Tanh()
    self.lstm = nn.LSTM(embedding_dim, hidden_size, num_hidden)
    self.fc = nn.Linear(2*num_hidden*hidden_size, qu_feature_size)

  def forward(self, question):
      # Two-layer LSTM → (h+c) for each → 2048-dim → FC → tanh → 1024-dim question embedding.
      embedded_question = self.embed_word(question)
      embedded_question = self.tanh(embedded_question)
      embedded_question = embedded_question.transpose(0, 1)
      _, (hidden, cell) = self.lstm(embedded_question)

      question_features = torch.cat((hidden, cell), dim=2)
      question_features = question_features.transpose(0, 1)
      question_features = question_features.reshape(question_features.size()[0], -1)
      question_features = self.tanh(question_features)
      question_features = self.fc(question_features)

      return question_features

class BaseLineVQA(nn.Module):

  def __init__(self, feature_size, que_vocab_size, ans_vocab_size, word_embed, hidden_size, num_hidden):
    super(BaseLineVQA, self).__init__()
    self.img_encoder = BaseLineImageEncoder(feature_size)
    self.que_encoder = BaseTextEncoder(que_vocab_size, word_embed, hidden_size, num_hidden, feature_size)
    self.dropout = nn.Dropout(0.5)
    self.tanh = nn.Tanh()
    self.fc1 = nn.Linear(feature_size, ans_vocab_size)
    self.fc2 = nn.Linear(ans_vocab_size, ans_vocab_size)
    # flagged

  def forward(self, image, question):
    img_features = self.img_encoder(image)
    que_features = self.que_encoder(question)

    combined = img_features * que_features
    combined = self.dropout(combined)
    combined = self.tanh(combined)
    combined = self.fc1(combined)
    combined = self.dropout(combined)
    combined = self.tanh(combined)
    logits = self.fc2(combined)
    return logits

#### Simple VQA Model
element wise multiplication (resnet with LSTM)

In [9]:
class resnetImageEncoder(nn.Module):

  def __init__(self, embed_dim):
    super(resnetImageEncoder, self).__init__()

    self.model = resnet50()
    self.model = nn.Sequential(*list(self.model.children())[:-1])
    self.flatten = nn.Flatten()
    self.fc = nn.Linear(2048, embed_dim)

  def forward(self, x):

      # x shape: [batch_size, 3, 224, 224]
      features = self.model(x)      # [batch_size, 2048, 1, 1]
      features = self.flatten(features)  # [batch_size, 2048]
      embedded = self.fc(features)  # [batch_size, embed_dim]
      return embedded


class SimpleVQA(nn.Module):

  def __init__(self, feature_size, que_vocab_size, ans_vocab_size, word_embed, hidden_size, num_hidden):
    super(SimpleVQA, self).__init__()
    self.img_encoder = resnetImageEncoder(feature_size)
    self.que_encoder = BaseTextEncoder(que_vocab_size, word_embed, hidden_size, num_hidden, feature_size)
    self.dropout = nn.Dropout(0.5)
    self.tanh = nn.Tanh()
    self.fc = nn.Linear(feature_size, ans_vocab_size)

  def forward(self, image, question):
    img_features = self.img_encoder(image)
    que_features = self.que_encoder(question)

    combined = img_features * que_features
    combined = self.dropout(combined)
    combined = self.tanh(combined)
    logits = self.fc(combined)
    return logits


#### Cross Modal Encoder
mixes between resnet and BERT

In [10]:
class SpatialImageEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(SpatialImageEncoder, self).__init__()
        vgg = models.vgg19(pretrained=True)
        self.features = vgg.features
        
        for param in self.features.parameters():
            param.requires_grad = False
        
        self.conv_dim = 512
        self.proj = nn.Linear(self.conv_dim, embed_dim)
    
    def forward(self, x):
        with torch.no_grad():
            features = self.features(x)  # (B, 512, H, W)
        
        B, C, H, W = features.shape
        features = features.view(B, C, H * W).transpose(1, 2)
        features = self.proj(features)
        features = nn.functional.normalize(features, p=2, dim=-1)
        return features  # (B, H*W, embed_dim)


class CrossAttentionBlock(nn.Module):
    def __init__(self, hidden_size, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=8,
            batch_first=True,
            dropout=dropout
        )
        self.ff = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 4, hidden_size)
        )
        self.norm1 = nn.LayerNorm(hidden_size)
        self.norm2 = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value):
        attn_out, _ = self.attn(query, key_value, key_value)
        attn_out = self.dropout(attn_out)
        out = self.norm1(query + attn_out)

        ff_out = self.ff(out)
        ff_out = self.dropout(ff_out)
        out = self.norm2(out + ff_out)
        return out

class CrossModalEncoderVQA(nn.Module):
    def __init__(self, feature_size, ans_vocab_size, hidden_size, dropout=0.1):
        super().__init__()
        
        self.img_encoder = SpatialImageEncoder(feature_size)
        self.img_proj = nn.Linear(feature_size, hidden_size)
        
        self.que_encoder = BertModel.from_pretrained('bert-base-uncased')
        # Optionally freeze BERT
        for param in self.que_encoder.parameters():
            param.requires_grad = False
            
        bert_dim = self.que_encoder.config.hidden_size
        if bert_dim != hidden_size:
            self.text_proj = nn.Linear(bert_dim, hidden_size)
        else:
            self.text_proj = nn.Identity()
        
        self.cross_attn = CrossAttentionBlock(hidden_size)
        self.dropout = nn.Dropout(dropout)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, ans_vocab_size)
        )
    
    def forward(self, images, input_ids, attention_mask=None):
        # Encode image
        img_feats = self.img_encoder(images)        # (B, num_regions, feature_size)
        img_feats = self.img_proj(img_feats)        # (B, num_regions, hidden_size)
        
        if attention_mask is None:
        # BERT's pad_token_id is 0
            attention_mask = (input_ids != 0).long()

        # Encode question
        with torch.no_grad():
            bert_out = self.que_encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        text_feats = bert_out.last_hidden_state     # (B, seq_len, bert_dim)
        text_feats = self.text_proj(text_feats)     # (B, seq_len, hidden_size)
        
        # Cross-modal fusion
        fused = self.cross_attn(query=text_feats, key_value=img_feats)
        pooled = fused[:, 0, :]                      # (B, hidden_size)
        pooled = self.dropout(pooled)
        
        # Answer prediction
        logits = self.classifier(pooled)
        return logits


#### TRAINER

In [11]:
import time

DATA_DIR = 'data'
BATCH_SIZE = 8
MAX_QUESTION_LENGTH = 32
LOG_DIR = 'log'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

class AverageMeter(object):
  def __init__(self):
    self.reset()

  def reset(self):
    self.val = 0
    self.avg = 0
    self.sum = 0
    self.count = 0

  def update(self, val, n=1):
    self.val = val
    self.sum += val * n
    self.count += n
    self.avg = self.sum / self.count


class Trainer:

  def __init__(self, model, lr, scheduler, epochs, model_name):

    self.model = model.to(device)
    self.lr = lr
    self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
    self.scheduler = scheduler(self.optimizer) if scheduler else None
    self.epochs = epochs

    self.best_val_loss = float('inf')
    self.current_epoch = 0
    self.model_name = model_name
    os.makedirs(os.path.join(LOG_DIR, self.model_name), exist_ok=True)

  def training_loop(self, dataloader):
    criterion = nn.CrossEntropyLoss()

    start_timing = time.time()
    for epoch in range(self.epochs):

      self.current_epoch = epoch
      train_loss_meter = AverageMeter()
      val_loss_meter = AverageMeter()

      self.model.train()

  #train
      loop = tqdm(dataloader['train'], desc=f"Epoch {epoch+1}/{self.epochs} [train]", leave=True)
      for idx, sample in enumerate(loop):

        image = sample['image'].to(device)
        question = sample['question'].to(device)
        label = sample['answer'].to(device)

        logits = self.model(image, question)
        loss = criterion(logits, label)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        train_loss_meter.update(loss.item(), image.size(0))
        loop.set_postfix(loss=train_loss_meter.avg)

  #val
      self.model.eval()
      loop = tqdm(dataloader['val'], desc=f"Epoch {epoch+1}/{self.epochs} [val]", leave=True)
      with torch.no_grad():
        for idx, sample in enumerate(loop):
          image = sample['image'].to(device=device)
          question = sample['question'].to(device=device)
          label = sample['answer'].to(device=device)
          logits = self.model(image, question)
          loss = criterion(logits, label)

          val_loss_meter.update(loss.item(), image.size(0))
          loop.set_postfix(loss=val_loss_meter.avg)

      train_loss = train_loss_meter.avg
      val_loss = val_loss_meter.avg

      # Log to files
      with open(os.path.join(os.path.join(LOG_DIR, self.model_name), f'{self.model_name}-train_log.txt'), 'a') as f:
          f.write(f"{epoch+1}\t{train_loss:.6f}\n")

      with open(os.path.join(os.path.join(LOG_DIR, self.model_name), f'{self.model_name}-val_log.txt'), 'a') as f:
          f.write(f"{epoch+1}\t{val_loss:.6f}\n")

      tqdm.write(
          f"Epoch {epoch+1}/{self.epochs} | "
          f"Train Loss: {train_loss:.6f} | "
          f"Val Loss: {val_loss:.6f}"
      )

      # ===== CHECKPOINTING =====
      if val_loss < self.best_val_loss:
          self.best_val_loss = val_loss
          self.save(os.path.join(os.path.join(LOG_DIR, self.model_name), f'{self.model_name}-best_model.pth'))
          tqdm.write(f"✓ Saved best model (val_loss: {val_loss:.6f})")

      # Save latest checkpoint
      self.save(os.path.join(os.path.join(LOG_DIR, self.model_name), f'{self.model_name}-latest_checkpoint.pth'))

      # Step scheduler
      if self.scheduler:
          self.scheduler.step()

    # ===== TRAINING COMPLETE =====
    stop_timing = time.time()
    timing = stop_timing - start_timing
    print(f"\n✓ Finished training: {timing//60:.0f}m {timing%60:.0f}s")
    print(f"✓ Best validation loss: {self.best_val_loss:.6f}")


  def save(self, path):

    torch.save({
        'epoch': self.current_epoch,
        'model_state_dict': self.model.state_dict(),
        'optimizer_state_dict': self.optimizer.state_dict(),
        'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
        'best_val_loss': self.best_val_loss,
    }, path)

  def load(self, path):

      checkpoint = torch.load(path, map_location=device)

      self.model.load_state_dict(checkpoint['model_state_dict'])
      self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

      if self.scheduler and checkpoint['scheduler_state_dict']:
          self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

      self.best_val_loss = checkpoint['best_val_loss']
      self.current_epoch = checkpoint['epoch'] + 1

      print(f"✓ Loaded checkpoint from epoch {checkpoint['epoch']}")
      print(f"✓ Best val loss: {self.best_val_loss:.6f}")

In [12]:
BATCH_SIZE = 128
MAX_QU_LEN = 30
NUM_WORKER = 4
FEATURE_SIZE, WORD_EMBED = 1024, 300
NUM_HIDDEN, HIDDEN_SIZE = 4, 512
LEARNING_RATE, STEP_SIZE, GAMMA = 0.001, 10, 0.1
EPOCH = 5


In [13]:
dataloader = data_loader(input_dir=DATA_DIR, batch_size=BATCH_SIZE, num_worker=4, max_length=MAX_QUESTION_LENGTH, npy_dir='regular')
que_vocab_size = dataloader['train'].dataset.qu_vocab.vocab_size
ans_vocab_size = dataloader["train"].dataset.ans_vocab.vocab_size


trainer = Trainer(
    model=BaseLineVQA(FEATURE_SIZE, que_vocab_size, ans_vocab_size, WORD_EMBED, HIDDEN_SIZE, NUM_HIDDEN ),
    lr=0.001,
    scheduler=lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
    epochs=5,
    model_name='baseline_model'
)

trainer.training_loop(dataloader)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 235MB/s]
Epoch 1/5 [val]: 100%|██████████| 1675/1675 [29:17<00:00,  1.05s/it, loss=2.78]


Epoch 1/5 | Train Loss: 3.390869 | Val Loss: 2.779350
✓ Saved best model (val_loss: 2.779350)


Epoch 2/5 [val]: 100%|██████████| 1675/1675 [29:19<00:00,  1.05s/it, loss=2.56]


Epoch 2/5 | Train Loss: 2.720277 | Val Loss: 2.562396
✓ Saved best model (val_loss: 2.562396)


Epoch 3/5 [val]: 100%|██████████| 1675/1675 [29:19<00:00,  1.05s/it, loss=2.47]


Epoch 3/5 | Train Loss: 2.564854 | Val Loss: 2.468216
✓ Saved best model (val_loss: 2.468216)


Epoch 4/5 [val]: 100%|██████████| 1675/1675 [29:20<00:00,  1.05s/it, loss=2.42]


Epoch 4/5 | Train Loss: 2.484271 | Val Loss: 2.418560
✓ Saved best model (val_loss: 2.418560)


Epoch 5/5 [val]: 100%|██████████| 1675/1675 [29:24<00:00,  1.05s/it, loss=2.39]


Epoch 5/5 | Train Loss: 2.428124 | Val Loss: 2.391010
✓ Saved best model (val_loss: 2.391010)

✓ Finished training: 472m 52s
✓ Best validation loss: 2.391010
